# Problem 1: 神经网络基础 (Neural Network Foundations)

在本问题中，你将实现构建 Transformer 语言模型所需的基础神经网络组件。

**限制**: 你不应该使用封装好的神经网络模块，如 `torch.nn.Linear`、`torch.nn.Embedding` 等，
而应该使用底层的张量操作（如 `torch.matmul`、索引等）来实现这些操作。

**目的**: 这将帮助你深入理解这些组件在底层是如何工作的。

## 1.1 Softmax

**目标**: 实现 `run_softmax` 函数。

Softmax 函数将一个向量归一化为概率分布。对于输入向量 $x \in \mathbb{R}^d$，softmax 函数定义为：

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_{j=1}^{d} e^{x_j}}$$

**实现要求**:
- 不能使用 `torch.nn.Softmax` 或 `torch.nn.functional.softmax`
- 需要处理数值稳定性问题（避免 exp 溢出）
- 支持对任意维度进行 softmax

**提示**: 可以使用指数技巧来提高数值稳定性：
$$\text{softmax}(x)_i = \frac{e^{x_i - \max(x)}}{\sum_{j=1}^{d} e^{x_j - \max(x)}}$$

**对应函数**: `tests/adapters.py` 中的 `run_softmax(in_features, dim)`

## 1.2 SiLU (Swish Activation)

**目标**: 实现 `run_silu` 函数。

SiLU (Sigmoid Linear Unit) 激活函数定义为：

$$\text{SiLU}(x) = x \cdot \sigma(x) = x \cdot \frac{1}{1 + e^{-x}}$$

其中 $\sigma(x)$ 是 sigmoid 函数。SiLU 也被称为 Swish 激活函数。

**实现要求**:
- 不能使用 `torch.nn.SiLU` 或 `torch.nn.functional.silu`
- 使用 sigmoid 函数的 PyTorch 实现（`torch.sigmoid`）是允许的

**对应函数**: `tests/adapters.py` 中的 `run_silu(in_features)`

## 1.3 线性层 (Linear Layers)

**目标**: 实现 `run_linear` 函数。

线性层（也称为全连接层或密集层）执行以下变换：

$$y = xW^T + b$$

其中：
- $x \in \mathbb{R}^{batch \times d_{in}}$ 是输入
- $W \in \mathbb{R}^{d_{out} \times d_{in}}$ 是权重矩阵
- $b \in \mathbb{R}^{d_{out}}$ 是偏置向量
- $y \in \mathbb{R}^{batch \times d_{out}}$ 是输出

**实现要求**:
- 不能使用 `torch.nn.Linear`
- 使用 `torch.matmul` 或 `@` 运算符实现矩阵乘法
- 需要处理批量输入（任意数量的前导维度）
- 偏置是可选的

**对应函数**: `tests/adapters.py` 中的 `run_linear(d_in, d_out, weights, in_features)`

**注意**: 在我们的参考实现中，偏置被包含在权重矩阵中（作为额外的一行/列）。

## 1.4 嵌入层 (Embedding Layers)

**目标**: 实现 `run_embedding` 函数。

嵌入层将离散的 token ID 映射到连续的向量表示。对于 token ID $i$，嵌入为：

$$\text{embedding}(i) = E[i]$$

其中 $E \in \mathbb{R}^{vocab\_size \times d_{model}}$ 是嵌入矩阵。

**实现要求**:
- 不能使用 `torch.nn.Embedding`
- 使用张量索引操作来获取嵌入向量
- 需要处理批量输入（任意形状的 token ID 张量）

**对应函数**: `tests/adapters.py` 中的 `run_embedding(vocab_size, d_model, weights, token_ids)`

**提示**: 可以直接使用 `weights[token_ids]` 这样的索引操作。

## 1.5 RMSNorm

**目标**: 实现 `run_rmsnorm` 函数。

RMSNorm (Root Mean Square Normalization) 是 LayerNorm 的一种简化版本。
对于输入 $x \in \mathbb{R}^d$，RMSNorm 定义为：

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \odot \gamma$$

其中：
- $\text{mean}(x^2) = \frac{1}{d}\sum_{i=1}^{d} x_i^2$ 是均方值
- $\epsilon$ 是一个小的常数，用于数值稳定性
- $\gamma \in \mathbb{R}^d$ 是可学习的尺度参数（weights）
- $\odot$ 表示逐元素乘法

**与 LayerNorm 的区别**: RMSNorm 不使用均值中心化（减去均值），只使用 RMS 缩放。

**实现要求**:
- 不能使用 `torch.nn.RMSNorm` 或 `torch.nn.LayerNorm`
- 需要处理任意形状的输入（保留前导维度）
- 沿着最后一个维度计算 RMS

**对应函数**: `tests/adapters.py` 中的 `run_rmsnorm(d_model, eps, weights, in_features)`

## 1.6 SwiGLU

**目标**: 实现 `run_swiglu` 函数。

SwiGLU 是一种用于 Transformer 前馈网络的激活函数变体。
对于输入 $x \in \mathbb{R}^{d_{model}}$，SwiGLU 定义为：

$$\text{SwiGLU}(x) = \text{SiLU}(W_1 x) \odot (W_3 x)$$

最终的输出为：

$$\text{output} = W_2 \cdot \text{SwiGLU}(x)$$

其中：
- $W_1 \in \mathbb{R}^{d_{ff} \times d_{model}}$ 是第一个线性变换的权重
- $W_2 \in \mathbb{R}^{d_{model} \times d_{ff}}$ 是输出投影的权重
- $W_3 \in \mathbb{R}^{d_{ff} \times d_{model}}$ 是门控线性变换的权重
- $d_{ff}$ 是隐藏层维度（通常 $d_{ff} > d_{model}$）
- $\odot$ 表示逐元素乘法

**实现步骤**:
1. 计算 $a = W_1 x$
2. 计算 $b = W_3 x$
3. 计算 $\text{SiLU}(a) \odot b$
4. 计算 $W_2 \cdot (\text{SiLU}(a) \odot b)$

**对应函数**: `tests/adapters.py` 中的 `run_swiglu(d_model, d_ff, w1_weight, w2_weight, w3_weight, in_features)`

**提示**: 你需要使用之前实现的 SiLU 函数（或者 PyTorch 的 `F.silu`）。